In [4]:
import numpy as np
import pandas as pd
import sys
sys.path.append('../')
from src.metrics.evaluator import HierarchicalPrevalenceEvaluator

In [3]:
# A realidade da nossa Bag (Ground Truth)
y_real_bag = [
  "A.B.C", "A.B.C", "A.B.C", "A.B.C", "A.B.C", # 50%
  "A.B.D", "A.B.D", "A.B.D",                   # 30%
  "A.E.F", "A.E.F"                             # 20%
]

# O que o nosso modelo hipotético previu (Prevalências Top-Down Ajustadas)
# Note que o erro aumenta conforme aprofundamos na árvore
prevalencias_estimadas = {
  # Nível 1: Acertou em cheio
  "A": 1.00,  
  
  # Nível 2: Errou por 0.10 (Era 80% A.B e 20% A.E)
  "A.B": 0.90, 
  "A.E": 0.10, 
  
  # Nível 3: Errou feio (Era 50% A.B.C, 30% A.B.D e 20% A.E.F)
  "A.B.C": 0.80,  # Previu 80% em vez de 50%
  "A.B.D": 0.10,  # Previu 10% em vez de 30%
  "A.E.F": 0.10   # Previu 10% em vez de 20%
}

In [8]:
avaliador = HierarchicalPrevalenceEvaluator(sep=".")

# Calculando...
erros_nivel, hmae = avaliador.evaluate_from_prevalences(y_real_bag, prevalencias_estimadas)

# Exibindo os resultados de forma elegante
print("--- ANÁLISE DE ERRO POR NÍVEL (MAE_L) ---")
for nivel, erro in erros_nivel.items():
  print(f"Nível {nivel}: {erro:.4f}")

print("\n--- ERRO HIERÁRQUICO PONDERADO ---")
print(f"hAE (Peso decrescente 1/L): {hmae:.4f}")

# (Opcional) Mostrando as prevalências lado a lado para conferência
print("\n--- TABELA DE COMPARAÇÃO (Real vs Predito) ---")
prev_reais_niveis = avaliador.calculate_true_prevalences(y_real_bag)

comparacao = []
for cls, pred_val in prevalencias_estimadas.items():
  nivel = len(cls.split("."))
  # Busca o valor real seguro (se a classe não existir no real, é 0.0)
  real_val = prev_reais_niveis.get(nivel, {}).get(cls, 0.0) 
  
  comparacao.append({
    "Nível": nivel,
    "Classe": cls,
    "Real (%)": f"{real_val*100:.1f}%",
    "Predito (%)": f"{pred_val*100:.1f}%",
    "Erro Absoluto": f"{abs(pred_val - real_val):.2f}"
  })

df_comparacao = pd.DataFrame(comparacao)
display(df_comparacao) # Usa o display nativo do Jupyter

--- ANÁLISE DE ERRO POR NÍVEL (MAE_L) ---
Nível 1: 0.0000
Nível 2: 0.2000
Nível 3: 0.6000

--- ERRO HIERÁRQUICO PONDERADO ---
hAE (Peso decrescente 1/L): 0.1636

--- TABELA DE COMPARAÇÃO (Real vs Predito) ---


,Nível,Classe,Real (%),Predito (%),Erro Absoluto
0,1,A,100.0%,100.0%,0.00
1,2,A.B,80.0%,90.0%,0.10
2,2,A.E,20.0%,10.0%,0.10
3,3,A.B.C,50.0%,80.0%,0.30
4,3,A.B.D,30.0%,10.0%,0.20
5,3,A.E.F,20.0%,10.0%,0.10
